# Quotes and Trades Contracts and Monitoring

Notebook refactorizado.

Ya no actua como lanzador de agentes.

Su funcion ahora es:
- ver el contrato de descarga,
- documentar comandos y rutas de prueba,
- dejar templates de produccion,
- y visualizar artefactos de certificacion para quotes y referencias relacionadas.

## Estado Actual

Este notebook sustituye al flujo antiguo que mezclaba:
- Agent01,
- Agent02 laxo `031`,
- Agent02 estricto `032`,
- y rutas antiguas `C:\TSIS_Data\data\quotes_p95`.

Operacion actual recomendada:
- lanzar Agent01/02/03 desde terminal,
- usar notebooks solo para consulta, contrato y visualizacion.

In [1]:
from pathlib import Path
import json
import pandas as pd

CONTRACT_FILE = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml")
CONTRACT_VIEWER = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\030_show_contract_polygon_download.py")

TEST_RUN_ID = "20260312_quotes_lifecycle3_test"
TEST_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / TEST_RUN_ID
TEST_QUOTES_ROOT = Path(r"D:\quotes\__pruebas__\lifecycle_prod")
TEST_TASKS_CSV = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3.csv"
TEST_TASKS_META = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3_meta.json"

PROD_QUOTES_ROOT = Path(r"D:\quotes")
PROD_RUN_DIR_TEMPLATE = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit")

print("CONTRACT_FILE:", CONTRACT_FILE, "exists", CONTRACT_FILE.exists())
print("CONTRACT_VIEWER:", CONTRACT_VIEWER, "exists", CONTRACT_VIEWER.exists())
print("TEST_RUN_DIR:", TEST_RUN_DIR)
print("TEST_QUOTES_ROOT:", TEST_QUOTES_ROOT)
print("PROD_QUOTES_ROOT:", PROD_QUOTES_ROOT)

CONTRACT_FILE: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml exists True
CONTRACT_VIEWER: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\030_show_contract_polygon_download.py exists True
TEST_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test
TEST_QUOTES_ROOT: D:\quotes\__pruebas__\lifecycle_prod
PROD_QUOTES_ROOT: D:\quotes


In [2]:
if CONTRACT_VIEWER.exists() and CONTRACT_FILE.exists():
    DATASET_TO_VIEW = "quotes"
    exec(CONTRACT_VIEWER.read_text(encoding="utf-8"), globals())
else:
    print("Contrato o viewer no disponibles")

Contract file: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml
version: 1 | profile: lax_realtime_coverage


,dataset,enabled,priority,target_path,expected_file_name,partition_pattern,min_cols,min_rows_per_file,coverage_unit,success_criteria
0,quotes,True,1,quotes,quotes.parquet,{ticker}/year={year}/month={month}/day={day}/q...,"timestamp, bid_price, ask_price",1,day,file_exists_and_rows_gt_0
1,trades_ticks_2005_2025,True,2,trades_ticks_2005_2025,market.parquet,{ticker}/year={year}/month={month}/day={day}/m...,"timestamp, price, size",1,day,file_exists_and_rows_gt_0



Detalle dataset: quotes
{
  "enabled": true,
  "priority": 1,
  "target_path": "quotes",
  "expected_file_name": "quotes.parquet",
  "partition_pattern": "{ticker}/year={year}/month={month}/day={day}/quotes.parquet",
  "partition_keys": [
    "ticker",
    "year",
    "month",
    "day"
  ],
  "minimum_required_columns": [
    "timestamp",
    "bid_price",
    "ask_price"
  ],
  "non_empty_required": true,
  "min_rows_per_file": 1,
  "soft_quality_rules": {
    "bid_ask_sanity": {
      "enabled": true,
      "expression": "bid_price <= ask_price",
      "allow_violations": true
    },
    "non_negative_prices": {
      "enabled": true,
      "columns": [
        "bid_price",
        "ask_price"
      ],
      "allow_violations": true
    }
  },
  "coverage": {
    "coverage_unit": "day",
    "expected_files_per_day": 1,
    "success_criteria": "file_exists_and_rows_gt_0",
    "completeness_metric": "present_days / expected_days_official_window"
  },
  "retry": {
    "enabled": true,


## Comandos de Prueba que se Han Usado

Corrida de prueba actual:
- `RUN_ID = 20260312_quotes_lifecycle3_test`
- `QuotesRoot = D:\quotes\__pruebas__\lifecycle_prod`
- `TasksCsv = C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv`

Artefactos principales:
- `C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test`
- `quotes_agent_strict_events_current.csv`
- `retry_queue_quotes_strict_current.csv`
- `live_status_quotes_strict.json`
- `agent03_outputs\run_summary.json`

In [3]:
from pathlib import Path

DOWNLOADER_SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py")
AGENT_RUNNERS_DIR = Path(globals().get("AGENT_RUNNERS_DIR", r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents"))
AGENT02_PS1 = Path(globals().get("AGENT02_PS1", AGENT_RUNNERS_DIR / "run_agent02_quotes_strict_loop.ps1"))
AGENT03_LIVE_PS1 = Path(globals().get("AGENT03_LIVE_PS1", AGENT_RUNNERS_DIR / "run_agent03_live_fast.ps1"))
AGENT03_COMPACT_PS1 = Path(globals().get("AGENT03_COMPACT_PS1", AGENT_RUNNERS_DIR / "run_agent03_monitor_compact.ps1"))

print(rf'''python {DOWNLOADER_SCRIPT} ^
  --csv "{TEST_TASKS_CSV}" ^
  --output "{TEST_QUOTES_ROOT}" ^
  --concurrent 24 ^
  --run-id "{TEST_RUN_ID}" ^
  --run-dir "{TEST_RUN_DIR}" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -QuotesRoot "{TEST_QUOTES_ROOT}" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_LIVE_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -IntervalSec 10''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_COMPACT_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -SleepSec 30''')

python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py ^
  --csv "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv" ^
  --output "D:\quotes\__pruebas__\lifecycle_prod" ^
  --concurrent 24 ^
  --run-id "20260312_quotes_lifecycle3_test" ^
  --run-dir "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -QuotesRoot "D:\quotes\__pruebas__\lifecycle_prod" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent03_live_fast.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -IntervalSec 10

powershell -Exec

## Comandos Preparados de Produccion

Quotes:
- raiz objetivo: `D:\quotes`
- mismo esquema de `RUN_ID` y `RUN_DIR`
- misma sesion extendida `04:00:00 -> 20:00:00 ET`

Trades:
- este notebook no deja un launcher de produccion de trades porque el flujo operativo actual no esta estandarizado aqui.
- se mantiene como referencia de contrato y monitoreo, no como orchestrator de `quotes + trades`.

In [4]:
PROD_RUN_ID = "YYYYMMDD_quotes_session_prod_01"
PROD_RUN_DIR = PROD_RUN_DIR_TEMPLATE / PROD_RUN_ID
PROD_TASKS_CSV = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\quotes_tasks_prod.csv")

print(rf'''python {DOWNLOADER_SCRIPT} ^
  --csv "{PROD_TASKS_CSV}" ^
  --output "{PROD_QUOTES_ROOT}" ^
  --concurrent 24 ^
  --run-id "{PROD_RUN_ID}" ^
  --run-dir "{PROD_RUN_DIR}" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -QuotesRoot "{PROD_QUOTES_ROOT}" ^
  -MaxFiles 50000 ^
  -SleepSec 15''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_LIVE_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -IntervalSec 10''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_COMPACT_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -SleepSec 30''')

python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py ^
  --csv "C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\quotes_tasks_prod.csv" ^
  --output "D:\quotes" ^
  --concurrent 24 ^
  --run-id "YYYYMMDD_quotes_session_prod_01" ^
  --run-dir "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -QuotesRoot "D:\quotes" ^
  -MaxFiles 50000 ^
  -SleepSec 15

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent03_live_fast.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -IntervalSec 10

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent03_monitor_compact.ps1" ^
  -RunId "

## Visualizacion de Resultados y Artefactos

Esta parte reemplaza a los launchers viejos de `031` y `032` dentro del notebook.

In [5]:
events_current = TEST_RUN_DIR / "quotes_agent_strict_events_current.csv"
retry_current = TEST_RUN_DIR / "retry_queue_quotes_strict_current.csv"
live_status = TEST_RUN_DIR / "live_status_quotes_strict.json"
run_summary = TEST_RUN_DIR / "agent03_outputs" / "run_summary.json"
causes_by_ticker = TEST_RUN_DIR / "agent03_outputs" / "causes_by_ticker.csv"

for p in [events_current, retry_current, live_status, run_summary, causes_by_ticker]:
    print(p, "exists", p.exists())

if live_status.exists():
    print("\nLIVE STATUS")
    print(json.dumps(json.loads(live_status.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if events_current.exists():
    ev = pd.read_csv(events_current)
    print("\nEVENTS CURRENT rows:", len(ev))
    if "severity" in ev.columns:
        print(ev["severity"].value_counts(dropna=False))
    display(ev.head(20))

if retry_current.exists():
    rq = pd.read_csv(retry_current)
    print("\nRETRY CURRENT rows:", len(rq))
    display(rq.head(20))

if run_summary.exists():
    print("\nRUN SUMMARY")
    print(json.dumps(json.loads(run_summary.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if causes_by_ticker.exists():
    cbt = pd.read_csv(causes_by_ticker)
    print("\nCAUSES BY TICKER rows:", len(cbt))
    display(cbt.head(20))

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\retry_queue_quotes_strict_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\live_status_quotes_strict.json exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\agent03_outputs\run_summary.json exists False
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\agent03_outputs\causes_by_ticker.csv exists False

LIVE STATUS
{
  "run_id": "20260312_quotes_lifecycle3_test",
  "updated_utc": "2026-03-12T14:02:00.388761+00:00",
  "probe_root": "D:\\quotes\\__pruebas__\\lifecycle_prod",
  "max_files": 50000,
  "files_discovered_total": 1695,
  "files_pending": 401,
  "files_processed_total_state"

,file,rows,severity,issues,warns,action,crossed_rows,crossed_ratio_pct,negative_price_rows,missing_required_cols,dtype_mismatches,ask_integer_pct,bid_integer_pct,ask_eq_round_bid_pct,processed_at_utc,run_id
0,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1493,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,7,0.468855,0,[],[],6.831882,4.621567,6.630944,2026-03-12 14:01:58.112045+00:00,20260312_quotes_lifecycle3_test
1,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,225,PASS,[],[],count_coverage,0,0.000000,0,[],[],2.666667,0.444444,2.666667,2026-03-12 14:01:58.113872+00:00,20260312_quotes_lifecycle3_test
2,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1259,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.158856,0,[],[],5.003971,1.906275,5.003971,2026-03-12 14:01:58.115725+00:00,20260312_quotes_lifecycle3_test
3,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,9700,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.020619,0,[],[],1.536082,5.886598,1.536082,2026-03-12 14:01:58.118402+00:00,20260312_quotes_lifecycle3_test
4,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,260,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.384615,0.769231,0.384615,2026-03-12 14:01:58.120017+00:00,20260312_quotes_lifecycle3_test
5,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1601,PASS,[],[],count_coverage,0,0.000000,0,[],[],1.374141,2.186134,1.311680,2026-03-12 14:01:58.121839+00:00,20260312_quotes_lifecycle3_test
6,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,659,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.435508,9.711684,7.283763,2026-03-12 14:01:58.123427+00:00,20260312_quotes_lifecycle3_test
7,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,399,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.017544,2.756892,7.017544,2026-03-12 14:01:58.124988+00:00,20260312_quotes_lifecycle3_test
8,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,442,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.452489,0.452489,0.226244,2026-03-12 14:01:58.126618+00:00,20260312_quotes_lifecycle3_test
9,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,273,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.732601,1.098901,0.366300,2026-03-12 14:01:58.128085+00:00,20260312_quotes_lifecycle3_test



RETRY CURRENT rows: 0


,file,severity,issues,warns,action,processed_at_utc
